# 📷 Digital Image Enhancement & Restoration Assignment
**Course:** Image Processing and Computer Vision (Module 2)  
**Instructor:** Dr. Guindo (DAUST)  
**Author:** Computer Vision Assignment  

---
## 🎯 Objectives & Core Concepts

1. **Enhancement vs. Restoration (Slide 2)**:
   - **Enhancement**: Subjective process to make an image more useful (brighter, clearer, sharper).
   - **Restoration**: Objective process to reverse a *known physical degradation* (noise, low contrast, blur). Evaluated objectively using **PSNR (dB)** and **Entropy**.
2. **Two Families of Operations (Slide 3)**:
   - **Point Operations ($s = T(r)$)**: Modifies individual pixels independently without looking at neighbors.
   - **Spatial Operations**: Modifies pixels using a local neighborhood/window surrounding each pixel.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.degradation import (
    infect_low_contrast, infect_low_light, 
    infect_salt_and_pepper, infect_overexposed
)
from src.point_transforms import (
    transform_negative, transform_linear, transform_minmax_stretch,
    transform_percentile_stretch, transform_log, transform_gamma,
    transform_threshold, extract_bit_planes
)
from src.histogram_ops import global_histogram_equalization, apply_clahe
from src.spatial_ops import (
    apply_mean_filter, apply_gaussian_filter, apply_median_filter,
    apply_unsharp_masking
)
from src.utils import calculate_stats, calculate_psnr

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
print('Environment and helper modules successfully imported!')

## 🖼️ Step 0: Load and Inspect Original Reference Image
We start by loading our input image `original_image.png` and calculating its statistical properties (Mean, Standard Deviation / Contrast, and Entropy in bits).

In [ ]:
orig_img = cv2.imread('original_image.png')
if orig_img is None:
    raise FileNotFoundError('original_image.png not found!')

orig_rgb = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
orig_stats = calculate_stats(orig_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(orig_rgb)
plt.title(f'Original Reference Image\nMean: {orig_stats["mean"]}, Std: {orig_stats["std"]}')
plt.axis('off')

plt.subplot(1, 2, 2)
gray_orig = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)
plt.hist(gray_orig.ravel(), bins=256, range=(0, 256), color='navy', alpha=0.7)
plt.title(f'Intensity Histogram\nEntropy: {orig_stats["entropy"]} bits')
plt.xlabel('Intensity')
plt.ylabel('Pixel Count')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🦠 Step 1: Image Degradation ("Infection") Forward Model
To test enhancement and restoration methods, we first synthesize four distinct real-world degradations (`degradation.py`):
1. **Low Contrast**: Squashes intensity range to $[25, 75]$ (similar to low-contrast *Image I* on Slide 5).
2. **Low Light (Underexposure)**: Scales brightness down by factor $0.15$.
3. **Impulse Noise**: Adds $4\%$ Salt-and-Pepper noise.
4. **Over-exposure**: Shifts intensities $+110$ with highlight clipping at $255$.

In [ ]:
inf_low_contrast = infect_low_contrast(orig_img, 25, 75)
inf_low_light = infect_low_light(orig_img, factor=0.15)
inf_salt_pepper = infect_salt_and_pepper(orig_img, amount=0.04)
inf_overexposed = infect_overexposed(orig_img, shift=110, scale=1.2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
infections = [
    ("1. Low Contrast", inf_low_contrast),
    ("2. Low Light", inf_low_light),
    ("3. Salt & Pepper Noise", inf_salt_pepper),
    ("4. Over-exposed", inf_overexposed)
]

for ax, (title, img) in zip(axes, infections):
    st = calculate_stats(img)
    psnr_val = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean:{st['mean']} Std:{st['std']}\nPSNR: {psnr_val} dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 1: Infected (Degraded) Image States', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Step 2: Point Operations — Contrast Stretching & Linear Gain (Slides 15-19)

* **Linear Contrast Stretching Formula**:
  $$s = 255 \times \frac{r - r_{\text{min}}}{r_{\text{max}} - r_{\text{min}}}$$
* **The One-Pixel Disaster (Slide 17)**: Ordinary min-max stretching fails if even 1 pixel is 0 or 255.
* **Robust Percentile Stretching (Slide 18-19)**: Clips values between 1st and 99th percentiles ($lo, hi$) to make the operation robust to outliers.

Let's see how the low-contrast infected image evolves after applying these techniques!

In [ ]:
linear_boost = transform_linear(inf_low_contrast, alpha=1.8, beta=-40)
minmax_stretched = transform_minmax_stretch(inf_low_contrast)
percentile_stretched = transform_percentile_stretch(inf_low_contrast, 1, 99)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
step2_images = [
    ("Infected Low Contrast", inf_low_contrast),
    ("Linear Boost (a=1.8, b=-40)", linear_boost),
    ("Min-Max Stretch", minmax_stretched),
    ("Percentile Stretch (1-99%)", percentile_stretched)
]

for idx, (title, img) in enumerate(step2_images):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    
    # Image view
    axes[0, idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, idx].set_title(f"{title}\nStd:{st['std']} | PSNR:{psnr_v}dB", fontsize=9)
    axes[0, idx].axis('off')
    
    # Histogram view
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[1, idx].hist(g.ravel(), bins=256, range=(0, 256), color='teal', alpha=0.7)
    axes[1, idx].set_xlim([0, 255])
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Step 2: Evolution from Low Contrast to Stretched Full Dynamic Range', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🌙 Step 3: Rescuing Details in Dark Shadows (Log & Gamma Transforms, Slides 20-24)

* **Log Transform**: $s = c \cdot \log(1 + r)$, expands low-intensity dark tones while compressing highlights.
* **Gamma Correction**: $s = 255 \cdot \left(\frac{r}{255}\right)^\gamma$
  * $\gamma < 1$ (e.g. $\gamma=0.4$): Brightens dark shadow detail.
  * $\gamma > 1$ (e.g. $\gamma=2.0$): Darkens over-exposed highlights.

In [ ]:
log_enhanced = transform_log(inf_low_light)
gamma_04 = transform_gamma(inf_low_light, gamma=0.4)
gamma_20 = transform_gamma(inf_overexposed, gamma=2.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
step3_list = [
    ("Infected: Low Light", inf_low_light),
    ("Log Transform s=c log(1+r)", log_enhanced),
    ("Gamma Correction (γ=0.4)", gamma_04),
    ("Overexposed -> Gamma(γ=2.0)", gamma_20)
]

for ax, (title, img) in zip(axes, step3_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean:{st['mean']} | PSNR:{psnr_v}dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 3: Rescuing Low-Light Shadows & Taming Over-Exposure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 Step 4: Histogram Equalization vs. CLAHE (Slides 29-45)

* **Global Equalization (HE)**: Maps CDF to produce a flat histogram across the entire image.
* **CLAHE (Contrast-Limited Adaptive Histogram Equalization)**: Equalizes local $8 \times 8$ tiles with a histogram clip limit to cap noise amplification.
* **CRITICAL RULE (Slide 43)**: *NEVER apply equalization to R, G, B channels separately!* We convert to **$Lab$ color space**, equalize the **$L$ (Lightness) channel ONLY**, and convert back.

In [ ]:
he_enhanced = global_histogram_equalization(inf_low_contrast)
clahe_enhanced = apply_clahe(inf_low_contrast, clip_limit=2.5, tile_grid_size=(8, 8))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
step4_list = [
    ("Low Contrast Input", inf_low_contrast),
    ("Global Equalization (HE)", he_enhanced),
    ("CLAHE (Lab L-channel)", clahe_enhanced)
]

for idx, (title, img) in enumerate(step4_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    
    axes[0, idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, idx].set_title(f"{title}\nStd:{st['std']} | Entropy:{st['entropy']}b\nPSNR:{psnr_v}dB", fontsize=9)
    axes[0, idx].axis('off')
    
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[1, idx].hist(g.ravel(), bins=256, range=(0, 256), color='purple', alpha=0.7)
    axes[1, idx].set_xlim([0, 255])
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Step 4: Global Histogram Equalization vs CLAHE (Lab Color Space)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🧹 Step 5: Spatial Neighborhood Filtering (Denoising & Sharpening, Slide 3)

Here we compare **Mean (Box)**, **Gaussian**, and **Median** filters on the Salt-and-Pepper noise infection, followed by **Unsharp Masking** sharpening.

In [ ]:
mean_denoised = apply_mean_filter(inf_salt_pepper, 5)
gauss_denoised = apply_gaussian_filter(inf_salt_pepper, 5, 1.2)
median_denoised = apply_median_filter(inf_salt_pepper, 5)
unsharp_sharpened = apply_unsharp_masking(median_denoised, 5, 1.0, 1.5)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
step5_list = [
    ("S&P Noise Input", inf_salt_pepper),
    ("Mean Filter (5x5)", mean_denoised),
    ("Gaussian Filter (5x5)", gauss_denoised),
    ("Median Filter (5x5)", median_denoised),
    ("Median + Unsharp Mask", unsharp_sharpened)
]

for ax, (title, img) in zip(axes, step5_list):
    st = calculate_stats(img)
    psnr_v = calculate_psnr(orig_img, img)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nPSNR: {psnr_v} dB", fontsize=9)
    ax.axis('off')

plt.suptitle('Step 5: Denoising Salt & Pepper Noise (Median Filter achieves +21.7 dB PSNR gain!)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔢 Step 6: Bit-Plane Slicing (Slides 26-27)

Decomposing pixels into 8 binary bit planes to visualize structural content vs noise floor.

In [ ]:
bit_planes = extract_bit_planes(orig_img)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for idx, (b_name, b_img) in enumerate(bit_planes.items()):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(b_img, cmap='gray')
    axes[row, col].set_title(b_name, fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Step 6: Bit-Plane Slicing (Top Bits 7-5 carry 88% signal; Bits 2-0 carry noise)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏆 Quantitative Comparison & Final Summary Table

Let's output the final comparison table evaluated objectively using **Mean**, **Contrast (Std Dev)**, **Entropy**, and **PSNR (dB)**.

In [ ]:
eval_list = [
    ("Original Reference", orig_img, orig_img),
    ("Infected: Low Contrast", inf_low_contrast, orig_img),
    ("Linear Boost (a=1.8, b=-40)", linear_boost, orig_img),
    ("Percentile Stretch (1-99%)", percentile_stretched, orig_img),
    ("Global Equalization (HE)", he_enhanced, orig_img),
    ("CLAHE (Lab L-channel)", clahe_enhanced, orig_img),
    ("Infected: Low Light", inf_low_light, orig_img),
    ("Log Transform", log_enhanced, orig_img),
    ("Gamma Correction (gamma=0.4)", gamma_04, orig_img),
    ("Infected: S&P Noise", inf_salt_pepper, orig_img),
    ("Mean Denoised (5x5)", mean_denoised, orig_img),
    ("Gaussian Denoised (5x5)", gauss_denoised, orig_img),
    ("Median Denoised (5x5)", median_denoised, orig_img),
]

print(f"{'Method/Image':<30} | {'Mean':<6} | {'Std (Contrast)':<14} | {'Entropy':<8} | {'PSNR (dB)':<9}")
print("-" * 70)
for name, img, ref in eval_list:
    st = calculate_stats(img)
    psnr_val = calculate_psnr(ref, img)
    print(f"{name:<30} | {st['mean']:<6} | {st['std']:<14} | {st['entropy']:<8} | {psnr_val:<9}")